In [1]:
import requests
import os
import sys
import platform
from lakehouse import bronze, silver
from pyspark.sql import DataFrame, SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
import json

In [2]:
if platform.system() == "Windows":
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    print("Adding Python ENV variables on Windows")

Adding Python ENV variables on Windows


In [3]:
builder = (
    SparkSession.builder.appName("Data with Nikk the Greek Spark Session")
    .master("local[4]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [4]:
CATALOG = spark.catalog.currentCatalog()

# 1. Set Up and Bronze Data

In [5]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

DataFrame[]

In [6]:
options = {"catalog": CATALOG, "target_schema": "bronze"}

In [7]:
@F.udf(returnType="STRING")
def get_properties(url):
    json_request = requests.get(url).json()
    return json.dumps(json_request["result"]["properties"])

In [8]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("properties", get_properties(F.col("url")))


bronze_instance = StarWarsBronze(spark, **options)

In [9]:
bronze_instance.load().transform().write(mode="overwrite").execute("people", "planets")

2025-02-28 00:09:12 | people | execute | Started
2025-02-28 00:09:12 | people | load | Started
2025-02-28 00:09:17 | people | load | Completed in 0.07 min
2025-02-28 00:09:17 | people | transform | Started
2025-02-28 00:09:17 | people | transform | Completed in 0.0 min
2025-02-28 00:09:17 | people | write | Started
2025-02-28 00:09:58 | people | write | Completed in 0.68 min
2025-02-28 00:09:58 | people | execute | Completed in 0.77 min
2025-02-28 00:09:58 | planets | execute | Started
2025-02-28 00:09:58 | planets | load | Started
2025-02-28 00:10:01 | planets | load | Completed in 0.03 min
2025-02-28 00:10:01 | planets | transform | Started
2025-02-28 00:10:01 | planets | transform | Completed in 0.0 min
2025-02-28 00:10:01 | planets | write | Started
2025-02-28 00:10:46 | planets | write | Completed in 0.73 min
2025-02-28 00:10:46 | planets | execute | Completed in 0.78 min


In [10]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 82
+--------------------+-------------------+---+--------------------+--------------------+
|         LH_BronzeTS|               name|uid|                 url|          properties|
+--------------------+-------------------+---+--------------------+--------------------+
|2025-02-28 00:09:...|        Cliegg Lars| 62|https://www.swapi...|{"created": "2025...|
|2025-02-28 00:09:...|  Poggle the Lesser| 63|https://www.swapi...|{"created": "2025...|
|2025-02-28 00:09:...|    Luminara Unduli| 64|https://www.swapi...|{"created": "2025...|
|2025-02-28 00:09:...|      Barriss Offee| 65|https://www.swapi...|{"created": "2025...|
|2025-02-28 00:09:...|              Dormé| 66|https://www.swapi...|{"created": "2025...|
|2025-02-28 00:09:...|              Dooku| 67|https://www.swapi...|{"created": "2025...|
|2025-02-28 00:09:...|Bail Prestor Organa| 68|https://www.swapi...|{"created": "2025...|
|2025-02-28 00:09:...|         Jango Fett| 69|https://www.swapi...|{"created": "2025...|
|2025-02

In [11]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.planets")
print(f"No. Rows: {df.count()}")
df.show(truncate=False)

No. Rows: 60
+--------------------------+--------------+---+-------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|LH_BronzeTS               |name          |uid|url                                  |properties                                                                                                                                                                                                                                                                                                                                                                                    |
+--------------------------+--------------+---+--

In [12]:
ddl_schema_planets = "diameter STRING, rotation_period STRING, orbital_period STRING, gravity STRING, population STRING, climate STRING, terrain STRING, surface_water STRING, created STRING, edited STRING, name STRING, url STRING"
ddl_schema_people = "height STRING, mass STRING, hair_color STRING, skin_color STRING, eye_color STRING, birth_year STRING, gender STRING, created STRING, edited STRING, name STRING, homeworld STRING, url STRING"
DDL_SCHEMAS = {"planets": ddl_schema_planets, "people": ddl_schema_people}

In [13]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

# 2 Overwrite

In [14]:
class StarWarsSilver(silver.Silver):
    def custom_filter(self, df: DataFrame, table: str) -> DataFrame:
        return df.where("uid <= '25'")

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        df = df.withColumn(
            "properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table])
        )
        df = df.withColumn("uid", F.col("uid").cast("int"))
        return df

    def add_dummy_col(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("dummy_col", F.lit("dummy"))


silver_instance = StarWarsSilver(
    spark, catalog=CATALOG, source_schema="bronze", target_schema="silver"
)

In [15]:
silver_instance = silver_instance.load(filter="custom")
silver_instance.transform(
    transformation_order=[
        "add_dummy_col",
        "rename_columns",
        "tbl_transformations",
        "select_columns",
        "cast_column_types",
    ],
    rename_columns={
        "planets": {"dummy_col": "dummy"},
        "people": {"dummy_col": "dummy"},
    },
    select_columns={
        "planets": ["LH_BronzeTS", "name", "uid", "url", "dummy"],
        "people": ["LH_BronzeTS", "name", "uid", "url", "dummy"],
    },
    cast_column_types={
        "planets": {"dummy": "string", "id": "int"},
        "people": {"dummy": "string", "id": "int"},
    },
)
silver_instance.write(mode="overwrite").execute("people", "planets")

2025-02-28 00:10:49 | people | execute | Started
2025-02-28 00:10:49 | people | load | Started
2025-02-28 00:10:49 | people | load | Completed in 0.0 min
2025-02-28 00:10:49 | people | transform | Started
2025-02-28 00:10:49 | people | transform | Completed in 0.0 min
2025-02-28 00:10:49 | people | write | Started
2025-02-28 00:10:52 | people | write | Completed in 0.03 min
2025-02-28 00:10:52 | people | execute | Completed in 0.03 min
2025-02-28 00:10:52 | planets | execute | Started
2025-02-28 00:10:52 | planets | load | Started
2025-02-28 00:10:52 | planets | load | Completed in 0.0 min
2025-02-28 00:10:52 | planets | transform | Started
2025-02-28 00:10:52 | planets | transform | Completed in 0.0 min
2025-02-28 00:10:52 | planets | write | Started
2025-02-28 00:10:53 | planets | write | Completed in 0.02 min
2025-02-28 00:10:53 | planets | execute | Completed in 0.02 min


In [16]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

No. Rows: 17
+--------------------------+--------------------------+---------------------+---+------------------------------------+-----+
|LH_SilverTS               |LH_BronzeTS               |name                 |uid|url                                 |dummy|
+--------------------------+--------------------------+---------------------+---+------------------------------------+-----+
|2025-02-28 00:10:50.021706|2025-02-28 00:09:18.185005|Luke Skywalker       |1  |https://www.swapi.tech/api/people/1 |dummy|
|2025-02-28 00:10:50.021706|2025-02-28 00:09:18.185005|C-3PO                |2  |https://www.swapi.tech/api/people/2 |dummy|
|2025-02-28 00:10:50.021706|2025-02-28 00:09:18.185005|Obi-Wan Kenobi       |10 |https://www.swapi.tech/api/people/10|dummy|
|2025-02-28 00:10:50.021706|2025-02-28 00:09:18.185005|Anakin Skywalker     |11 |https://www.swapi.tech/api/people/11|dummy|
|2025-02-28 00:10:50.021706|2025-02-28 00:09:18.185005|Wilhuff Tarkin       |12 |https://www.swapi.tech/api/peop

In [17]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.planets")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

No. Rows: 18
+--------------------------+--------------------------+--------------+---+-------------------------------------+-----+
|LH_SilverTS               |LH_BronzeTS               |name          |uid|url                                  |dummy|
+--------------------------+--------------------------+--------------+---+-------------------------------------+-----+
|2025-02-28 00:10:52.242761|2025-02-28 00:10:01.771848|Mygeeto       |16 |https://www.swapi.tech/api/planets/16|dummy|
|2025-02-28 00:10:52.242761|2025-02-28 00:10:01.771848|Felucia       |17 |https://www.swapi.tech/api/planets/17|dummy|
|2025-02-28 00:10:52.242761|2025-02-28 00:10:01.771848|Cato Neimoidia|18 |https://www.swapi.tech/api/planets/18|dummy|
|2025-02-28 00:10:52.242761|2025-02-28 00:10:01.771848|Saleucami     |19 |https://www.swapi.tech/api/planets/19|dummy|
|2025-02-28 00:10:52.242761|2025-02-28 00:10:01.771848|Stewjon       |20 |https://www.swapi.tech/api/planets/20|dummy|
|2025-02-28 00:10:52.242761|2025-02

# 6 Clean Up

In [18]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.silver CASCADE")

DataFrame[]